# CISim：人工内耳（CI）聴取音シミュレーション

人工内耳（Cochlear Implant, CI）利用者に聞こえている音を、**ノイズバンドボコーダ**と**電流拡散モデル（G行列）**で再現するノートブックです。
さらに、電流拡散（チャンネル間のにじみ）を **逆行列による事前補償** で打ち消せるかを検証しています。

## 処理の全体像

```
入力音声 → ① 22chバンドパス分割 → ② 包絡線抽出 → ③ 電流拡散（G行列） → ④ ノイズキャリアで再合成 → 出力音声
                                                     ↑
                                         （逆補償：G⁻¹ を事前に掛ける）
```

## 目次

| # | セクション | 内容 |
|---|---|---|
| 0 | 環境準備 | ライブラリのインストール・読み込み |
| 1 | 電流拡散モデル（G行列） | 電極間の電流のにじみを行列で表現・可視化 |
| 2 | FFT版シミュレーション（初期版） | 周波数領域でチャンネルパワーをにじませる試作 |
| 3 | フィルタバンク版ボコーダ（本命） | 時間領域で22chボコーダを構成 |
| 4 | 補助セル | 包絡線の可視化、旧版の再合成関数 |
| 5 | 逆補償の検証 | G⁻¹ による事前補償と正則化パラメータの比較 |
| 6 | 音楽ファイルへの適用 | mp3を入力にした一連の処理（単体で実行可能） |
| 7 | 整理版スクリプト | 入力パターン・キャリア切替に対応した最新版（単体で実行可能） |

> **注意**：各セルは試行錯誤の順に書かれているため、同じ名前の関数（`resynthesize` など）が**引数の違う形で再定義**されています。
> 上から順に実行すると後のセルの定義で上書きされるので、各セクションの注意書きを参照してください。
> セクション6・7は単体で完結しているので、まず動かすならこの2つがおすすめです。

---
## 0. 環境準備

必要なライブラリ（`soundfile`, `librosa`, `scipy`）をインストールします。初回のみ実行すれば十分です。

In [ ]:
!pip install soundfile librosa scipy

使用するライブラリを読み込みます。

- `numpy`：数値計算・行列演算
- `scipy`：信号処理（フィルタ、ヒルベルト変換）
- `matplotlib`：グラフ描画
- `librosa` / `soundfile`：音声ファイルの読み書き

※ `urllib.request` は現在のコードでは使用していません。

In [ ]:
import urllib.request
import numpy as np
import scipy
import matplotlib.pyplot as plt
import librosa
import soundfile as sf
print("Done")

---
## 1. 電流拡散モデル（G行列）

CIでは電極から流した電流が蝸牛内で広がり、**隣接する電極の位置にも届いてしまいます**（チャンネル間相互作用）。
これを次の指数減衰モデルで表した 22×22 の行列を **G行列** と呼んでいます。

$$
G_{jk} = \exp\left(-\frac{|x_j - x_k|}{\lambda}\right)
$$

- $x_j$：電極 $j$ の位置 [mm]（電極間隔 0.75 mm、Cochlear社準拠で22電極）
- $\lambda$：電流の広がり（空間定数）。大きいほど「にじみ」が大きい
  - モノポーラ刺激：$\lambda = 9$ mm（文献値 8–11 mm の中央値）
  - バイポーラ刺激：$\lambda = 3$ mm（文献値 2–4 mm の中央値）

このセルでは `make_G()` を定義し、2種類の刺激モードでG行列をヒートマップで可視化します。
対角成分は常に1、隣の電極への影響はモノポーラ≈0.92、バイポーラ≈0.78 となります。

In [ ]:
# ===== パラメータ設定 =====
n_electrodes = 22          # 電極数(Cochlear社準拠)
electrode_spacing = 0.75   # 電極間隔(mm) Cochlear社の目安
lambda_mp = 9.0            # λ: モノポーラ刺激(8-11mmの中央値)
lambda_bp = 3.0            # λ: バイポーラ刺激(2-4mmの中央値)

# 電極の位置をmm単位で定義
x = np.arange(n_electrodes) * electrode_spacing  # [0, 0.75, 1.5, ..., 15.75] mm

# ===== G行列を作る関数 =====
def make_G(x, lam):
    """
    G[j,k] = exp(-|x_j - x_k| / λ)
    電極kを刺激したとき、電極jの位置にどれだけ電流が届くか
    """
    n = len(x)
    G = np.zeros((n, n))
    for j in range(n):
        for k in range(n):
            G[j, k] = np.exp(-abs(x[j] - x[k]) / lam)       
    return G

# G行列を作成
G_mp = make_G(x, lambda_mp)  # モノポーラ(にじみが大きい)
G_bp = make_G(x, lambda_bp)  # バイポーラ(にじみが小さい)

# ===== 可視化 =====
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

im1 = axes[0].imshow(G_mp, cmap='hot', vmin=0, vmax=1)
axes[0].set_title(f'G_Matrix (Monopolar, λ={lambda_mp}mm)')
axes[0].set_xlabel('stimulating electrode k')
axes[0].set_ylabel('affected electrode j')
plt.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(G_bp, cmap='hot', vmin=0, vmax=1)
axes[1].set_title(f'G_Matrix (bipolar, λ={lambda_bp}mm)')
axes[1].set_xlabel('stimulating electrode k')
axes[1].set_ylabel('affected electrode j')
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.show()

print("G行列のサイズ:", G_mp.shape)
print("対角成分(自分への影響):", G_mp[0,0])  # 常に1.0
print("隣の電極への影響(モノポーラ):", G_mp[0,1])
print("隣の電極への影響(バイポーラ):", G_bp[0,1])

---
## 2. FFT版シミュレーション（初期版・自作）

周波数領域で処理する最初の試作版です。

1. `sample.wav` を読み込み、FFTでスペクトルに変換
2. 188–7939 Hz を対数等間隔に22チャンネルへ分割し、各チャンネルの平均振幅（`channel_power`）を算出
3. 正規化したG行列を掛けてチャンネルパワーをにじませる（λ = 2, 5, 9 mm の3条件）
4. 位相をランダムにしてIFFTで時間波形に戻し、`CI_*.wav` として保存
5. 元スペクトルと処理後スペクトル、チャンネルパワーの比較をプロット

**この方式の限界**：音声全体を1回のFFTで処理するため、包絡線の**時間変化が失われます**。これがセクション3のフィルタバンク版に移行した理由です。

> ⚠️ 入力に `sample.wav` が必要です。

In [ ]:
#人工内耳（CI)から聞こえる音声の再現（自己作成版）
"""
sr_sine = 16000
duration = 2.0
t  = np.linspace(0 , duration , int(sr_sine * duration))
audio_sine = 0.5 * np.sin(2 * np.pi * 440 * t )
"""

audio_data , sr = sf.read('sample.wav')
print("shape:", audio_data.shape)
print("sr:", sr)

if audio_data.ndim == 2:
    audio_data = audio_data[:, 0]
    print("ステレオ→モノラルに変換しました")
# ===== 仮のパラメータ =====
#sr = 16000        # サンプリングレート
N = len(audio_data)    # サンプル数

# ===== フーリエ変換 =====
spectrum = np.fft.rfft(audio_data) # 複素数の配列
freqs_bin = np.fft.rfftfreq(N, 1/sr) # 各ビンの周波数(Hz)

# freqs_bin は例えば [0, 1, 2, 3, ... 8000] Hz のような配列
# spectrum[i] が freqs_bin[i] Hz の成分
freq_min = 188
freq_max = 7939
n_channels = 22

boundaries = np.logspace(np.log10(freq_min),np.log10(freq_max),n_channels + 1)

channel_bins = []
for i in range(n_channels):
    low = boundaries[i]
    high = boundaries[i+1]

    indices = np.where((freqs_bin >= low) & (freqs_bin < high))[0]
    channel_bins.append(indices)

channel_power = np.zeros(n_channels)
for i in range(n_channels):
    if len(channel_bins[i]) > 0:
        channel_power[i] =  np.mean(np.abs(spectrum[channel_bins[i]]))



G_sharp  = make_G(x, 2.0)  # にじみ小
G_medium = make_G(x, 5.0)  # にじみ中
G_wide   = make_G(x, 9.0)  # にじみ大

#phase = np.angle(spectrum)
random_phase = np.random.uniform(0 , 2 * np.pi , len(spectrum))
amplitude = np.abs(spectrum)

for name, G in [("sharp(λ=2)", G_sharp), ("midium(λ=5)",    G_medium), ("rarge(λ=9)", G_wide)]:
    G_norm = G / np.sum(G, axis=1, keepdims=True)
    spread_power = G_norm @ channel_power
    new_spectrum = np.zeros(len(spectrum),dtype=complex)
    for i in range(n_channels):
        if len(channel_bins[i]) > 0:
            for bin_idx in channel_bins[i]:
                new_spectrum[bin_idx] = spread_power[i] * np.exp(1j * random_phase[bin_idx])
                #new_spectrum[bin_idx] = spread_power[i] * np.exp(1j * phase[bin_idx])
    
    output = np.fft.irfft(new_spectrum , n=N)
    output = output / np.max(np.abs(output) + 1e-10)
    sf.write(f'CI_{name}.wav' , output , sr)
    print(f'Done : CI_{name}.wav')

print("spread_power_shape:",spread_power.shape)
print("spread_power[i]_shape:",spread_power[0])




plt.figure(figsize = (12,4))
plt.plot(freqs_bin, np.abs(spectrum), label = "Original",alpha = 0.7)
plt.plot(freqs_bin, np.abs(new_spectrum), label = "Sharp(λ=2)",alpha = 0.7)
plt.xlim(0,3000)
plt.xlabel("Frequency (Hz)")
plt.ylabel("Amplitude")
plt.title("Spread effection")
plt.legend()
plt.show()

plt.figure(figsize=(12, 4))
plt.bar(range(n_channels),channel_power,alpha = 0.7,label = "Original_spread_power")
plt.bar(range(n_channels), spread_power, alpha=0.7, label="Proceed_spread_power")
plt.xlabel("Channel_number")
plt.ylabel("dB")
plt.title("Procecced_channel_power")
plt.legend()
plt.show()        

---
## 3. フィルタバンク版ボコーダ

実際のCIの信号処理に近い、時間領域のノイズバンドボコーダです。入力は 700 Hz・50 ms の正弦波（コメントアウトでインパルスや `sample.wav` に切替可能）。

| ステップ | 処理 | 実装 |
|---|---|---|
| 1 | **帯域分割**：22chのバンドパスフィルタで分割 | 4次Butterworth + `lfilter` |
| 2 | **包絡線抽出**：各chの振幅の時間変化を取り出す | ヒルベルト変換の絶対値 → 400 Hz ローパス |
| 3 | **電流拡散**：包絡線をG行列でにじませる | 行和で正規化した $G$ との行列積 |
| 4 | **再合成**：各chの帯域ノイズ（キャリア）に包絡線を掛けて合計 | `make_carriers()` + `resynthesize()` |

- キャリア（帯域ノイズ）は1回だけ生成し、3条件（λ = 2, 5, 9 mm）で共通に使うことで、条件間の差がG行列だけによるものになるようにしています。
- 出力：`CI_sharp.wav` / `CI_medium.wav` / `CI_wide.wav`、および入力波形と出力波形（λ = 2）の比較グラフ

> ここで定義される `resynthesize(spread_envelopes, carriers)` は**2引数版**です。セクション4・5との違いに注意してください。

In [ ]:
#人工内耳（CI)から聞こえる音声の再現（AIで修正）
import numpy as np
import scipy.signal as signal
import soundfile as sf
import matplotlib.pyplot as plt

# ===== 音声読み込み =====
#audio_data, sr = sf.read('sample.wav')
#if audio_data.ndim == 2:
    #audio_data = audio_data[:, 0]


sr = 44100              
duration = 0.05         
n_samples = int(sr * duration)
time = np.arange(n_samples) / sr

#audio_data = np.zeros(n_samples)
target_freq = 700.0    


audio_data = np.sin(2 * np.pi * target_freq * time)
#impulse_idx = int(0.010 * sr)
#audio_data[impulse_idx] = 1.0

# ===== パラメータ =====
n_channels = 22
freq_min = 188
freq_max = 7939
envelope_cutoff = 400  # 包絡線の低域カットオフ(Hz)

# チャンネルの境界周波数
boundaries = np.logspace(np.log10(freq_min), np.log10(freq_max), n_channels + 1)

# ===== ステップ1: バンドパスフィルタで22チャンネルに分割 =====
channel_signals = []
for i in range(n_channels):
    low  = boundaries[i]   / (sr / 2)
    high = boundaries[i+1] / (sr / 2)
    low  = np.clip(low,  0.001, 0.999)
    high = np.clip(high, 0.001, 0.999)
    if low >= high:
        channel_signals.append(np.zeros_like(audio_data))
        continue
    b, a = signal.butter(4, [low, high], btype='band')
    filtered = signal.lfilter(b, a, audio_data)
    channel_signals.append(filtered)

channel_signals = np.array(channel_signals)  # shape: (22, サンプル数)
print("チャンネル分割完了:", channel_signals.shape)

# ===== ステップ2: 包絡線を抽出(時間変化する振幅) =====
def extract_envelope(sig, sr, cutoff=400):
    # ヒルベルト変換で瞬時振幅を取得
    envelope = np.abs(signal.hilbert(sig))
    # 低域フィルタで平滑化
    b, a = signal.butter(2, cutoff / (sr / 2), btype='low')
    return signal.lfilter(b, a, envelope)

envelopes = np.array([extract_envelope(ch, sr, envelope_cutoff) 
                      for ch in channel_signals])
print("包絡線抽出完了:", envelopes.shape)

# ===== ステップ3: G行列で電流拡散を適用 =====
n_electrodes = 22
electrode_spacing = 0.75
x = np.arange(n_electrodes) * electrode_spacing

def make_G(x, lam):
    n = len(x)
    G = np.zeros((n, n))
    for j in range(n):
        for k in range(n):
            G[j, k] = np.exp(-abs(x[j] - x[k]) / lam)
    return G

G_sharp  = make_G(x, 2.0)
G_medium = make_G(x, 5.0)
G_wide   = make_G(x, 9.0)

def apply_spread(envelopes, G):
    # G_norm: 各行の合計が1になるよう正規化
    G_norm = G / np.sum(G, axis=1, keepdims=True)
    # 行列積: shape (22, サンプル数)
    return G_norm @ envelopes

def make_carriers(boundaries, sr, n_samples):
    n_channels = len(boundaries) - 1
    carriers = np.zeros((n_channels, n_samples))  # ← carriers で統一
    for i in range(n_channels):
        noise = np.random.randn(n_samples)
        low  = boundaries[i]   / (sr / 2)
        high = boundaries[i+1] / (sr / 2)
        low  = np.clip(low,  0.001, 0.999)
        high = np.clip(high, 0.001, 0.999)
        if low >= high:
            continue
        b, a = signal.butter(4, [low, high], btype='band')
        carriers[i] = signal.lfilter(b, a, noise)
    return carriers

def resynthesize(spread_envelopes, carriers):
    output = np.sum(spread_envelopes * carriers, axis=0)
    return output

# キャリアは1回だけ作る(3条件で共通)
carriers = make_carriers(boundaries, sr, len(audio_data))

# ===== 3条件で出力 =====
for name, G, filename in [
    ("sharp",  G_sharp,  "CI_sharp.wav"),
    ("medium", G_medium, "CI_medium.wav"),
    ("wide",   G_wide,   "CI_wide.wav"),
]:
    spread_env = apply_spread(envelopes, G)
    output = resynthesize(spread_env, carriers)
    output = output / np.max(np.abs(output) + 1e-10)
    sf.write(filename, output, sr)
    print(f"Done: {filename}")

# 入出力
spread_env = apply_spread(envelopes, G_sharp)
out_sharp = resynthesize(spread_env, carriers)
out_sharp = out_sharp / np.max(np.abs(out_sharp) + 1e-10)

time = np.arange(len(audio_data)) / sr

# 2段のグラフを作成
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)

# 1. 元の入力波形
axes[0].plot(time, audio_data, color='black', lw=0.8)
axes[0].set_title('Original Input Waveform')
axes[0].set_ylabel('Amplitude')
axes[0].grid(True, alpha=0.3)

# 2. 出力波形 (Sharp)
axes[1].plot(time, out_sharp, color='tab:blue', lw=0.8)
axes[1].set_title('Output')
axes[1].set_xlabel('Time (sec)')
axes[1].set_ylabel('Amplitude')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


---
## 4. 補助セル

### 4-1. 包絡線の可視化

セクション3で抽出した包絡線のうち、ch1, 6, 11, 16, 22 の5本を重ねて表示し、チャンネルごとの振幅の時間変化を確認します。
（セクション3の実行後に実行してください）

In [ ]:
# ===== 包絡線の可視化部分 =====
plt.figure(figsize=(12, 4))
time = np.arange(len(audio_data)) / sr
for i in [0, 5, 10, 15, 21]:
    plt.plot(time, envelopes[i], label=f"ch{i+1}({boundaries[i]:.0f}Hz)", alpha=0.7)
plt.xlabel("time (sec)")
plt.ylabel("Amplitude")
plt.title("each_envelopes")
plt.legend()
plt.show()

### 4-2. 再合成関数（旧版）

キャリアを**関数内で毎回生成する**旧版の `resynthesize(spread_envelopes, boundaries, sr, n_samples)` です（4引数・ゼロ位相の `filtfilt` を使用）。

> ⚠️ このセルを実行すると、セクション3の2引数版 `resynthesize` が**上書き**されます。
> - セクション5-2（逆補償）は**2引数版**を使う → このセルを実行していると動きません
> - セクション5-3（正則化の比較）は**この4引数版**を使う

In [ ]:
# ===== ノイズキャリアに包絡線を掛けて再合成する部分 =====
def resynthesize(spread_envelopes, boundaries, sr, n_samples):
    output = np.zeros(n_samples)
    for i in range(len(spread_envelopes)):
        # そのチャンネルの中心周波数のホワイトノイズをキャリアにする
        noise = np.random.randn(n_samples)
        low  = boundaries[i]   / (sr / 2)
        high = boundaries[i+1] / (sr / 2)
        low  = np.clip(low,  0.001, 0.999)
        high = np.clip(high, 0.001, 0.999)
        if low >= high:
            continue
        b, a = signal.butter(4, [low, high], btype='band')
        carrier = signal.filtfilt(b, a, noise)
        # キャリアに包絡線を掛ける
        output += spread_envelopes[i] * carrier
    return output

---
## 5. 逆補償の検証

電流拡散が $G$ で表せるなら、刺激前に $G^{-1}$ を掛けておけば $G G^{-1} = I$ となり、にじみを打ち消せるはず、という考えを検証します。

### 5-1. 逆行列の計算と確認

`make_G_inv()` で $G$ の逆行列を計算します。そのままだと数値的に不安定になりうるため、対角に小さな値 $\varepsilon$ を足す**正則化**を行っています。

$$
G^{-1}_{\text{reg}} = (G + \varepsilon I)^{-1}
$$

$G \, G^{-1}$ が単位行列に近いかを表示して確認します。

> ⚠️ 保存されている出力の `NameError: G_sharp is not defined` は、このセルを単体で実行したためです（実行番号が1になっています）。先にセクション3を実行してください。

In [ ]:
#逆行列の確認
import numpy as np

# ===== G⁻¹を計算する =====
def make_G_inv(G, regularization=1e-3):
    """
    Gの逆行列を計算する
    regularization: 数値的に不安定になるのを防ぐための正則化パラメータ
    小さいほど理論的に正確、大きいほど安定するが補償が甘くなる
    """
    n = G.shape[0]
    # ティホノフ正則化: (G^T G + λI)^{-1} G^T
    # 単純な逆行列より数値的に安定
    G_inv = np.linalg.inv(G + regularization * np.eye(n))
    return G_inv

G_sharp_inv  = make_G_inv(G_sharp)
G_medium_inv = make_G_inv(G_medium)
G_wide_inv   = make_G_inv(G_wide)

print("G_sharp_inv shape:", G_sharp_inv.shape)

# G @ G_inv が単位行列に近いか確認
product = G_wide @ G_wide_inv
print("\nG_wide @ G_wide_inv (単位行列に近いほど良い):")
print(np.round(product, 2))

### 5-2. 逆補償の適用

`apply_spread_with_inv()` では、行和で正規化したG行列 $G_{\text{norm}}$ の逆行列を求め、

1. 包絡線に $G_{\text{norm}}^{-1}$ を掛ける（事前補償）
2. その結果に $G_{\text{norm}}$ を掛ける（実際のにじみ）

という順に処理します。3条件それぞれで次の2種類を保存します。

| ファイル | 内容 |
|---|---|
| `CI_*_compensated.wav` | 事前補償 → にじみ を通した最終的な聞こえ（理想的には元のCI音と同じ） |
| `CI_*_precomp.wav` | 事前補償だけを掛けた状態（電極に流す刺激に相当） |

> このセルは**2引数版** `resynthesize`（セクション3）を前提にしています。

In [ ]:
#逆補償部分
def apply_spread_with_inv(envelopes, G):
    """
    G_invで事前補償してからGでにじませる
    理想的には G @ G_inv = I(単位行列)なので元に戻るはず
    """
    # 正しい逆補償の適用手順
    G_norm = G / np.sum(G, axis=1, keepdims=True)

    # 正規化済みの行列に対して逆行列を計算する（外側で割る必要はなくなる）
    G_inv_true = np.linalg.inv(G_norm)

    # これで計算すると、理論上はにじみが完全に相殺される
    precompensated = G_inv_true @ envelopes
    spread = G_norm @ precompensated
    
    return spread, precompensated

# 3条件で比較
for name, G, filename_comp, filename_pre in [
    ("sharp",  G_sharp,  "CI_sharp_compensated.wav",  "CI_sharp_precomp.wav"),
    ("medium", G_medium,  "CI_medium_compensated.wav", "CI_medium_precomp.wav"),
    ("wide",   G_wide,   "CI_wide_compensated.wav",   "CI_wide_precomp.wav"),
]:
    spread_comp, precomp = apply_spread_with_inv(envelopes, G)
    
    # 逆補償あり(G_inv → G を通した後)
    output_comp = resynthesize(spread_comp, carriers)
    output_comp = output_comp / np.max(np.abs(output_comp) + 1e-10)
    sf.write(filename_comp, output_comp, sr)
    print(f"Done: {filename_comp}")
    
    # 事前補償だけかけた状態(Gを通す前)
    output_pre = resynthesize(precomp, carriers)
    output_pre = output_pre / np.max(np.abs(output_pre) + 1e-10)
    sf.write(filename_pre, output_pre, sr)
    print(f"Done: {filename_pre}")

print("\n比較用ファイル一覧:")
print("元の音声の模擬CI音  → CI_sharp/medium/wide.wav")
print("逆補償後の模擬CI音  → CI_sharp/medium/wide_compensated.wav")
print("事前補償だけの状態  → CI_sharp/medium/wide_precomp.wav")

### 5-3. 正則化パラメータの比較

λ = 9 mm（にじみ大）の条件で、正則化パラメータ $\varepsilon$ を $10^{-1}, 10^{-2}, 10^{-3}, 10^{-4}$ と変えて逆補償を行い、`CI_wide_reg*.wav` として保存します。
$\varepsilon$ が小さいほど理論上は正確に補償できますが、数値的に不安定になりやすくなります。

なお、このセルは 5-2 と異なり、正則化した逆行列を**さらに行和で正規化**してから使っています。

> このセルは**4引数版** `resynthesize`（セクション4-2）を前提にしています。

In [ ]:
for reg in [1e-1, 1e-2, 1e-3, 1e-4]:
    G_inv = make_G_inv(G_wide, regularization=reg)
    G_norm     = G_wide / np.sum(G_wide, axis=1, keepdims=True)
    G_inv_norm = G_inv  / np.sum(G_inv,  axis=1, keepdims=True)
    
    precomp = G_inv_norm @ envelopes
    spread  = G_norm @ precomp
    
    output = resynthesize(spread, boundaries, sr, len(audio_data))
    output = output / np.max(np.abs(output) + 1e-10)
    sf.write(f'CI_wide_reg{reg}.wav', output, sr)
    print(f"Done: CI_wide_reg{reg}.wav")

---
## 6. 音楽ファイルへの適用（単体で実行可能）

`jingle_21.mp3` を 16 kHz・モノラルで読み込み、セクション3〜5の処理を一通り行う**自己完結したセル**です。
関数（`make_G`, `make_G_inv`, `extract_envelope`, `apply_spread`, `resynthesize`）もすべてこのセル内で定義しています。

主な違い：
- フィルタは2次Butterworth + `filtfilt`（ゼロ位相）
- フィルタ出力に NaN / inf が出たチャンネルはゼロで代替（安定性対策）
- `resynthesize` は4引数版（キャリアを関数内で生成）

出力ファイル（聴き比べ用）：

| ファイル | 内容 |
|---|---|
| `jingle_21.mp3` | 元の音楽 |
| `music_CI_sharp.wav` | λ = 2 mm（にじみ小） |
| `music_CI_medium.wav` | λ = 5 mm（中間） |
| `music_CI_wide.wav` | λ = 9 mm（にじみ大） |
| `music_CI_wide_compensated.wav` | λ = 9 mm + 逆補償あり |

In [ ]:
import librosa
import numpy as np
import scipy.signal as signal
import soundfile as sf
import matplotlib.pyplot as plt

# ===== mp3読み込み(16000Hzにダウンサンプリング) =====
audio_data, sr = librosa.load('jingle_21.mp3', sr=16000, mono=True)
print("shape:", audio_data.shape)
print("sr:", sr)
print("最大値:", np.max(np.abs(audio_data)))
print("長さ:", len(audio_data)/sr, "秒")

# ===== パラメータ =====
n_channels = 22
freq_min = 188
freq_max = 7939
envelope_cutoff = 400

boundaries = np.logspace(np.log10(freq_min), np.log10(freq_max), n_channels + 1)

# ===== G行列 =====
n_electrodes = 22
electrode_spacing = 0.75
x = np.arange(n_electrodes) * electrode_spacing

def make_G(x, lam):
    n = len(x)
    G = np.zeros((n, n))
    for j in range(n):
        for k in range(n):
            G[j, k] = np.exp(-abs(x[j] - x[k]) / lam)
    return G

def make_G_inv(G, regularization=1e-3):
    n = G.shape[0]
    G_inv = np.linalg.inv(G + regularization * np.eye(n))
    return G_inv

G_sharp  = make_G(x, 2.0)
G_medium = make_G(x, 5.0)
G_wide   = make_G(x, 9.0)
G_wide_inv = make_G_inv(G_wide)

# ===== ステップ1: バンドパスフィルタで22チャンネルに分割 =====
channel_signals = []
for i in range(n_channels):
    low  = boundaries[i]   / (sr / 2)
    high = boundaries[i+1] / (sr / 2)
    low  = np.clip(low,  0.001, 0.999)
    high = np.clip(high, 0.001, 0.999)
    if low >= high or (high - low) < 0.001:
        channel_signals.append(np.zeros_like(audio_data))
        continue
    try:
        b, a = signal.butter(2, [low, high], btype='band')
        filtered = signal.filtfilt(b, a, audio_data)
        if np.any(np.isnan(filtered)) or np.any(np.isinf(filtered)):
            channel_signals.append(np.zeros_like(audio_data))
            print(f"ch{i+1}: 不安定 → ゼロで代替")
        else:
            channel_signals.append(filtered)
    except:
        channel_signals.append(np.zeros_like(audio_data))
        print(f"ch{i+1}: エラー → ゼロで代替")

channel_signals = np.array(channel_signals)
print("チャンネル分割完了:", channel_signals.shape)

# ===== ステップ2: 包絡線抽出 =====
def extract_envelope(sig, sr, cutoff=400):
    envelope = np.abs(signal.hilbert(sig))
    b, a = signal.butter(2, cutoff / (sr / 2), btype='low')
    return signal.filtfilt(b, a, envelope)

envelopes = np.array([extract_envelope(ch, sr, envelope_cutoff)
                      for ch in channel_signals])
print("包絡線抽出完了:", envelopes.shape)

# ===== ステップ3: G行列適用 =====
def apply_spread(envelopes, G):
    G_norm = G / np.sum(G, axis=1, keepdims=True)
    return G_norm @ envelopes

# ===== ステップ4: 再合成 =====
def resynthesize(spread_envelopes, boundaries, sr, n_samples):
    output = np.zeros(n_samples)
    for i in range(len(spread_envelopes)):
        noise = np.random.randn(n_samples)
        low  = boundaries[i]   / (sr / 2)
        high = boundaries[i+1] / (sr / 2)
        low  = np.clip(low,  0.001, 0.999)
        high = np.clip(high, 0.001, 0.999)
        if low >= high:
            continue
        b, a = signal.butter(2, [low, high], btype='band')
        carrier = signal.filtfilt(b, a, noise)
        output += spread_envelopes[i] * carrier
    return output

N = len(audio_data)

# ===== 出力: 3条件 + 逆補償 =====
for name, G, filename in [
    ("sharp",  G_sharp,  "music_CI_sharp.wav"),
    ("medium", G_medium, "music_CI_medium.wav"),
    ("wide",   G_wide,   "music_CI_wide.wav"),
]:
    spread = apply_spread(envelopes, G)
    output = resynthesize(spread, boundaries, sr, N)
    output = output / np.max(np.abs(output) + 1e-10)
    sf.write(filename, output, sr)
    print(f"Done: {filename}")

# 逆補償あり
G_norm     = G_wide / np.sum(G_wide,     axis=1, keepdims=True)
G_inv_norm = G_wide_inv / np.sum(G_wide_inv, axis=1, keepdims=True)
precomp     = G_inv_norm @ envelopes
spread_comp = G_norm @ precomp
output_comp = resynthesize(spread_comp, boundaries, sr, N)
output_comp = output_comp / np.max(np.abs(output_comp) + 1e-10)
sf.write("music_CI_wide_compensated.wav", output_comp, sr)
print("Done: music_CI_wide_compensated.wav")

print("\n聴き比べ順番:")
print("1. jingle_21.mp3                  → 元の音楽")
print("2. music_CI_sharp.wav             → λ=2(にじみ小)")
print("3. music_CI_medium.wav            → λ=5(中間)")
print("4. music_CI_wide.wav              → λ=9(にじみ大)")
print("5. music_CI_wide_compensated.wav  → 逆補償あり")

---
## 7. 整理版スクリプト（単体で実行可能・最新版）

セクション3を整理し直した最新版です。上から順に処理が並んでいて、このセルだけで完結します。

**入力信号の選択**（コメントアウトで切替）
- パターンA：1000 Hz・50 ms の正弦波（デフォルト）
- パターンB：10 ms の位置に置いた単位インパルス
- パターンC：`sample.wav` の読み込み

**処理の流れ**
1. 22chバンドパスフィルタで帯域分割（因果的な `lfilter`）
2. ヒルベルト変換 + 400 Hz ローパスで包絡線抽出
3. G行列で電流拡散（`make_G` はブロードキャストでループなしに計算）
4. キャリアを生成して再合成
   - `mode='noise'`：帯域ノイズ（ノイズボコーダ）
   - `mode='tone'`：各chの中心周波数（下限と上限の幾何平均）の正弦波（サイン波ボコーダ）
5. λ = 2 mm の条件で入力波形と出力波形を比較プロット（時間軸は ms）

In [ ]:
import numpy as np
import scipy.signal as signal
import soundfile as sf
import matplotlib.pyplot as plt

# ==========================================
# 1. 入力信号の準備（いずれかを選択）
# ==========================================

# --- パターンA: 特定周波数の正弦波（デフォルト） ---
sr = 44100              # サンプリング周波数 (Hz)
duration = 0.05         # 信号長 (50 ms)
target_freq = 1000.0    # 周波数 (Hz)
n_samples = int(sr * duration)
time = np.arange(n_samples) / sr
audio_data = np.sin(2 * np.pi * target_freq * time)

# --- パターンB: 単位インパルス信号 ---
# sr = 44100
# duration = 0.05
# n_samples = int(sr * duration)
# time = np.arange(n_samples) / sr
# audio_data = np.zeros(n_samples)
# impulse_idx = int(0.010 * sr)  # 10ms の位置に配置
# audio_data[impulse_idx] = 1.0

# --- パターンC: 音声ファイルの読み込み ---
# audio_data, sr = sf.read('sample.wav')
# if audio_data.ndim == 2:
#     audio_data = audio_data[:, 0]
# time = np.arange(len(audio_data)) / sr

# ==========================================
# 2. パラメータ設定
# ==========================================
n_channels = 22
freq_min = 188
freq_max = 7939
envelope_cutoff = 400   # 包絡線LPFのカットオフ (Hz)

# 帯域分割境界周波数（対数等間隔）
boundaries = np.logspace(np.log10(freq_min), np.log10(freq_max), n_channels + 1)

# ==========================================
# 3. 信号処理関数群（因果的処理: lfilter）
# ==========================================

# ステップ1: バンドパスフィルタによる帯域分割
channel_signals = []
for i in range(n_channels):
    low  = boundaries[i]     / (sr / 2)
    high = boundaries[i + 1] / (sr / 2)
    low  = np.clip(low,  0.001, 0.999)
    high = np.clip(high, 0.001, 0.999)
    
    if low >= high:
        channel_signals.append(np.zeros_like(audio_data))
        continue
        
    b, a = signal.butter(4, [low, high], btype='band')
    filtered = signal.lfilter(b, a, audio_data)
    channel_signals.append(filtered)

channel_signals = np.array(channel_signals)  # shape: (22, n_samples)

# ステップ2: ヒルベルト変換とLPFによる包絡線抽出
def extract_envelope(sig, sr, cutoff=400):
    envelope = np.abs(signal.hilbert(sig))
    b, a = signal.butter(2, cutoff / (sr / 2), btype='low')
    return signal.lfilter(b, a, envelope)

envelopes = np.array([extract_envelope(ch, sr, envelope_cutoff) for ch in channel_signals])

# ステップ3: 電流拡散モデル (G行列)
electrode_spacing = 0.75  # 電極間隔 (mm)
x = np.arange(n_channels) * electrode_spacing

def make_G(x, lam):
    return np.exp(-np.abs(x[:, None] - x[None, :]) / lam)

G_sharp  = make_G(x, 2.0)
G_medium = make_G(x, 5.0)
G_wide   = make_G(x, 9.0)

def apply_spread(envs, G):
    G_norm = G / np.sum(G, axis=1, keepdims=True)
    return G_norm @ envs

# ステップ4: キャリア生成（ノイズボコーダ / サイン波ボコーダ切替可能）
def make_carriers(boundaries, sr, n_samples, mode='noise'):
    n_ch = len(boundaries) - 1
    carriers = np.zeros((n_ch, n_samples))
    t = np.arange(n_samples) / sr
    
    for i in range(n_ch):
        if mode == 'noise':
            # ノイズキャリア
            noise = np.random.randn(n_samples)
            low  = np.clip(boundaries[i]     / (sr / 2), 0.001, 0.999)
            high = np.clip(boundaries[i + 1] / (sr / 2), 0.001, 0.999)
            if low >= high:
                continue
            b, a = signal.butter(4, [low, high], btype='band')
            carriers[i] = signal.lfilter(b, a, noise)
        elif mode == 'tone':
            # サイン波キャリア（幾何平均の中心周波数）
            center_freq = np.sqrt(boundaries[i] * boundaries[i + 1])
            carriers[i] = np.sin(2 * np.pi * center_freq * t)
            
    return carriers

def resynthesize(spread_envs, carriers):
    return np.sum(spread_envs * carriers, axis=0)

# キャリア生成 (mode='noise' または mode='tone')
carriers = make_carriers(boundaries, sr, len(audio_data), mode='noise')

# ==========================================
# 4. G_sharp 条件での再合成と正規化
# ==========================================
spread_env = apply_spread(envelopes, G_sharp)
out_sharp = resynthesize(spread_env, carriers)
out_sharp = out_sharp / np.max(np.abs(out_sharp) + 1e-10)

# 音声ファイルとして保存する場合
# sf.write('output_sharp.wav', out_sharp, sr)

# ==========================================
# 5. 入出力波形のプロット
# ==========================================
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)

# 1段目: 入力波形
axes[0].plot(time * 1000, audio_data, color='black', lw=1.0)
axes[0].set_title('Input Signal')
axes[0].set_ylabel('Amplitude')
axes[0].grid(True, alpha=0.3)

# 2段目: 出力波形 (Sharp)
axes[1].plot(time * 1000, out_sharp, color='tab:blue', lw=0.8)
axes[1].set_title('Output Signal: Sharp (Spread λ = 2.0 mm)')
axes[1].set_xlabel('Time (ms)')
axes[1].set_ylabel('Amplitude')
axes[1].grid(True, alpha=0.3)

# 表示範囲を絞る場合（例: 0〜25 ms）
# plt.xlim(0, 25)

plt.tight_layout()
plt.show()